# WiLoR → LSC50: Reemplazo de landmarks de manos

Este notebook procesa los videos COLOR_BODY del dataset LSC50 con WiLoR (CVPR 2025)  
y genera CSVs de landmarks de manos en el mismo formato que usa el proyecto SignAI.

**Flujo:**
1. Videos mp4/avi → WiLoR → joints 3D (21 puntos por mano)
2. Conversión de orden MANO → MediaPipe
3. Normalización de coordenadas al formato CSV del proyecto
4. Guardar en Google Drive

**Requiere antes de ejecutar:**
- Cuenta en https://mano.is.tue.mpg.de/ para descargar `MANO_RIGHT.pkl`
- Videos COLOR_BODY subidos a Google Drive
- Runtime de Colab con GPU (Menú → Entorno de ejecución → Cambiar tipo → T4 GPU)

**Estado (2026-09-18):** el desfase de posición de los landmarks quedó resuelto (faltaba
recentrar los joints respecto a la muñeca antes de proyectar — ver Celda 7) y validado en
3 señas (ABUELO, TÍO, HOLA). Listo para correr sobre el corpus completo.

**Estructura:** Celdas 1-10 son el flujo principal (instalación → generar CSVs). El
Apéndice al final documenta cómo se diagnosticó y corrigió el desfase — no hace falta
correrlo para generar el corpus, queda como referencia.

## Celda 1 — Instalar WiLoR y dependencias (~3 min)

In [ ]:
# Clonar repositorio
!git clone --recursive https://github.com/rolpotamias/WiLoR.git
%cd WiLoR

# PyTorch con CUDA (ya viene en Colab, solo verificar)
import torch
print(f'CUDA disponible: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NINGUNA — cambiar runtime a GPU"}')

# Dependencias de WiLoR
!pip install -q -e .
!pip install -q ultralytics opencv-python-headless pandas tqdm

## Celda 1b — Instalar dependencias adicionales (chumpy, smplx, pytorch-lightning, yacs, pyrender, omegaconf)

In [ ]:
# La Celda 1 solo instala el paquete WiLoR en modo editable + unas pocas libs
# (ultralytics, opencv, pandas, tqdm). El resto de imports que usa este notebook
# (Celda 5 en adelante: wilor.models, wilor.utils, ViTDetDataset...) necesita lo
# de acá — se consolida todo en un solo lugar para no andar instalando a mitad
# del notebook cada vez que un import falla.

# setuptools>=60 trae un shim que reemplaza el "distutils" del stdlib (eliminado
# en Python 3.12+, PEP 632) por su propia copia vendorizada. Se fija <82 porque
# (a) torch 2.11 exige setuptools<82, y (b) setuptools>=82 quitó ese shim de
# distutils otra vez, así que "latest" rompe lo mismo que estamos arreglando.
!pip install -q -U "setuptools>=68,<82" wheel

# chumpy usa "from distutils.core import setup" en su propio setup.py.
# --no-build-isolation lo compila con el setuptools ya fijado arriba (que sí
# tiene el shim) en vez de un entorno de build aislado que se trae su propia
# copia de setuptools (típicamente la última, sin el shim).
!pip install -q --no-build-isolation "chumpy @ git+https://github.com/mattloper/chumpy"

# smplx (usado por wilor/models/mano_wrapper.py), pytorch-lightning y yacs
# (usados por wilor/models/wilor.py) y pyrender (usado por wilor/utils/renderer.py,
# que se importa automáticamente al hacer `from wilor.utils import recursive_to`).
# Del requirements.txt oficial del repo, esto es lo único que hace falta para los
# imports de este notebook — verificado grepeando el repo clonado. El resto
# (xtcocotools, hydra*, pyrootutils, rich, webdataset, gradio) es para
# entrenamiento o para gradio_demo.py, no se usa acá, y xtcocotools ni siquiera
# compila en Python 3.13 (Colab actual) — instalar solo lo necesario evita
# arrastrar builds rotos de paquetes que no se van a usar.
!pip install -q smplx==0.1.28 pytorch-lightning yacs pyrender

# omegaconf: usado en la Celda 5 para cargar model_config.yaml.
!pip install -q omegaconf

# wilor/utils/renderer.py ya configura PYOPENGL_PLATFORM='egl' por defecto si no
# está seteada — esto usa la GPU de Colab para renderizado offscreen sin X11.
# Si algo de esto falla con un error de EGL/display, descomentar para forzar
# renderizado por software (más lento):
# import os
# os.environ['PYOPENGL_PLATFORM'] = 'osmesa'
# !apt-get install -y libosmesa6-dev freeglut3-dev

## Celda 2 — Descargar pesos del modelo

In [ ]:
import os
os.makedirs('pretrained_models', exist_ok=True)

!wget -q --show-progress \
    https://huggingface.co/spaces/rolpotamias/WiLoR/resolve/main/pretrained_models/detector.pt \
    -O pretrained_models/detector.pt

!wget -q --show-progress \
    https://huggingface.co/spaces/rolpotamias/WiLoR/resolve/main/pretrained_models/wilor_final.ckpt \
    -O pretrained_models/wilor_final.ckpt

!wget -q --show-progress \
    https://huggingface.co/spaces/rolpotamias/WiLoR/resolve/main/pretrained_models/model_config.yaml \
    -O pretrained_models/model_config.yaml

print('Pesos descargados:', os.listdir('pretrained_models'))

## Celda 3 — Subir modelos MANO

1. Ir a https://mano.is.tue.mpg.de/ → registrarse → Downloads
2. Descargar **MANO v1.2** (viene un zip con `MANO_RIGHT.pkl` y `MANO_LEFT.pkl` adentro)
3. Ejecutar la celda y seleccionar **los dos archivos a la vez** en el selector que aparece

In [ ]:
from google.colab import files
import shutil, os

os.makedirs('mano_data', exist_ok=True)

print('Selecciona MANO_RIGHT.pkl Y MANO_LEFT.pkl (los dos a la vez en el selector):')
uploaded = files.upload()  # puedes seleccionar múltiples archivos a la vez

for fname in uploaded:
    dest = f'mano_data/{fname}'
    shutil.move(fname, dest)
    print(f'Guardado: {dest}')

# Verificar que están los dos
expected = {'MANO_RIGHT.pkl', 'MANO_LEFT.pkl'}
found = set(os.listdir('mano_data'))
missing = expected - found
if missing:
    print(f'\nFALTA: {missing}')
    print('Vuelve a ejecutar esta celda y sube el archivo que falta.')
else:
    print('\nOK — ambos modelos MANO presentes.')

## Celda 4 — Montar Google Drive y configurar rutas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── AJUSTAR ESTAS RUTAS ──────────────────────────────────────────────────────
# Carpeta con los videos COLOR_BODY (mp4 o avi)
# Estructura esperada: cada archivo se llama SSSS_VVVV_RRRR.mp4
VIDEOS_DIR = '/content/drive/MyDrive/lsc50/videos'

# Carpeta donde se guardarán los CSVs nuevos
# Se crearán subcarpetas LEFT_HAND_LANDMARKS/ y RIGHT_HAND_LANDMARKS/
OUTPUT_DIR = '/content/drive/MyDrive/lsc50/HANDS_WILOR'
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(f'{OUTPUT_DIR}/LEFT_HAND_LANDMARKS', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/RIGHT_HAND_LANDMARKS', exist_ok=True)

videos = sorted([f for f in os.listdir(VIDEOS_DIR) if f.endswith(('.mp4', '.avi'))])
print(f'Videos encontrados: {len(videos)}')
print('Primeros 5:', videos[:5])

## Celda 5 — Cargar modelo WiLoR

In [ ]:
import sys, os, shutil
sys.path.insert(0, '/content/WiLoR')

import torch
import numpy as np
import cv2
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from ultralytics import YOLO

from wilor.utils import recursive_to
from wilor.datasets.vitdet_dataset import ViTDetDataset, DEFAULT_MEAN, DEFAULT_STD

# cam_crop_to_full inline — evita importar wilor.utils.renderer (requiere pyrender)
def cam_crop_to_full(cam_bbox, box_center, box_size, img_size, focal_length=5000.):
    img_w, img_h = img_size[:, 0], img_size[:, 1]
    cx, cy = box_center[:, 0], box_center[:, 1]
    bs = box_size * cam_bbox[:, 0] + 1e-9
    tz = 2 * focal_length / bs
    tx = (2 * cam_bbox[:, 1] + (cx - img_w / 2.) * 2.) / bs
    ty = (2 * cam_bbox[:, 2] + (cy - img_h / 2.) * 2.) / bs
    return np.stack([tx, ty, tz], axis=-1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando: {device}')

detector = YOLO('pretrained_models/detector.pt')

from wilor.models.wilor import WiLoR as WiLoRModel
from omegaconf import OmegaConf

model_cfg = OmegaConf.load('pretrained_models/model_config.yaml')

# vitpose backbone no se necesita en inferencia — ya viene en el checkpoint
model_cfg.MODEL.BACKBONE.PRETRAINED_WEIGHTS = None

# El config espera los PKL en mano_data/mano/ — crear subcarpeta y copiar
os.makedirs('mano_data/mano', exist_ok=True)
for pkl in ['MANO_RIGHT.pkl', 'MANO_LEFT.pkl']:
    src, dst = f'mano_data/{pkl}', f'mano_data/mano/{pkl}'
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst)

# mano_mean_params.npz — parámetros medios de pose requeridos por el modelo
# (la carpeta en el repo se llama mano_data/, no data/ — esa ruta no existe)
if not os.path.exists('mano_data/mano_mean_params.npz'):
    !wget -q --show-progress \
        https://github.com/rolpotamias/WiLoR/raw/main/mano_data/mano_mean_params.npz \
        -O mano_data/mano_mean_params.npz

model = WiLoRModel(model_cfg)
ckpt = torch.load('pretrained_models/wilor_final.ckpt', map_location='cpu', weights_only=False)
model.load_state_dict(ckpt['state_dict'], strict=False)
model = model.to(device).eval()
print('Modelo WiLoR cargado.')


## Celda 6 — Funciones de conversión

Convierte del orden de joints MANO al orden de MediaPipe Hands.

In [ ]:
# WiLoR ya devuelve los 21 joints en el mismo orden que OpenPose/MediaPipe Hands:
# el remapeo mano_to_openpose = [0,13,14,15,16, 1,2,3,17, 4,5,6,18, 10,11,12,19, 7,8,9,20]
# ya se aplica DENTRO del modelo (wilor/models/mano_wrapper.py, clase MANO.forward).
# 0=Wrist, 1-4=Thumb(CMC,MCP,IP,TIP), 5-8=Index, 9-12=Middle, 13-16=Ring, 17-20=Pinky
# — el mismo orden que usa MediaPipe Hands. NO hay que reordenar otra vez aquí:
# hacerlo revolvería los dedos (bug encontrado en la primera versión de este notebook).

CSV_COLS = [f'landmark_{i}_{c}' for i in range(21) for c in ['x', 'y', 'z']]


def joints_to_mediapipe_row(joints_3d, kpts_2d, img_w, img_h):
    """
    Convierte joints WiLoR (ya en orden MediaPipe) a fila CSV.

    joints_3d : (21, 3) — coordenadas 3D en espacio de cámara (WiLoR), ya
                con el espejo de mano izquierda aplicado si corresponde.
    kpts_2d   : (21, 2) — proyección 2D en píxeles, misma corrección aplicada.
    img_w, img_h: dimensiones de la imagen original

    Retorna lista de 63 floats: [x0,y0,z0, x1,y1,z1, ..., x20,y20,z20]
    en coordenadas MediaPipe normalizadas.
    """
    x = kpts_2d[:, 0] / img_w   # [0, 1]
    y = kpts_2d[:, 1] / img_h   # [0, 1], y=0 en la parte superior

    # z: profundidad relativa a la muñeca en espacio canónico MANO (metros).
    # NO dividir por img_w: z_raw (~0.01-0.05 m entre falanges adyacentes)
    # ya está en la misma escala numérica que x,y normalizados de MediaPipe.
    # Dividir por img_w (~640) produce z ~1000x demasiado pequeño → dedos siempre extendidos.
    z_raw = joints_3d[:, 2] - joints_3d[0, 2]   # relativo a muñeca
    z = z_raw                         # escala correcta: metros MANO ≈ MediaPipe normalized

    row = []
    for i in range(21):
        row.extend([float(x[i]), float(y[i]), float(z[i])])
    return row


EMPTY_ROW = [0.0] * 63  # fila cuando no hay detección (igual que MediaPipe)

print('Funciones de conversión listas (sin remapeo manual — WiLoR ya entrega orden MediaPipe).')


## Celda 7 — Procesar videos

In [ ]:
BATCH_SIZE = 8   # frames por batch en WiLoR (reducir a 4 si hay OOM)
RESCALE_FACTOR = 2.0  # factor de ViTDetDataset (ver demo.py original de WiLoR)


def process_video(video_path):
    """
    Procesa un video y retorna (rows_left, rows_right) donde cada uno
    es una lista de filas CSV, una por frame.
    """
    cap = cv2.VideoCapture(str(video_path))
    img_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    img_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    rows_L, rows_R = [], []

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Detección de manos con YOLO
        det_out = detector(img_rgb, verbose=False)
        boxes = det_out[0].boxes

        row_L = EMPTY_ROW[:]
        row_R = EMPTY_ROW[:]

        if boxes is not None and len(boxes) > 0:
            bboxes  = boxes.xyxy.cpu().numpy()
            # is_right: 1 si la clase detectada es mano derecha, 0 si es izquierda
            # (confirmado contra demo.py oficial: det.boxes.cls == 1 → derecha)
            is_right_arr = boxes.cls.cpu().numpy().astype(int)

            # Preparar dataset para WiLoR
            dataset = ViTDetDataset(
                model_cfg,
                img_rgb,
                bboxes,
                right=is_right_arr,
                rescale_factor=RESCALE_FACTOR
            )
            dataloader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

            for batch in dataloader:
                batch = recursive_to(batch, device)
                with torch.no_grad():
                    out = model(batch)

                # Extraer resultados por detección
                pred_keypoints_3d = out['pred_keypoints_3d'].cpu().numpy()  # (N, 21, 3)
                pred_cam           = out['pred_cam'].cpu().numpy()           # (N, 3)
                is_right_batch     = batch['right'].cpu().numpy()            # (N,)
                box_center         = batch['box_center'].cpu().numpy()       # (N, 2)
                box_size           = batch['box_size'].cpu().numpy()         # (N,)
                img_size_batch     = batch['img_size'].cpu().numpy()         # (N, 2)

                for n in range(len(pred_keypoints_3d)):
                    joints_3d  = pred_keypoints_3d[n].copy()  # (21, 3)
                    is_right   = bool(is_right_batch[n])
                    multiplier = 1.0 if is_right else -1.0

                    # WiLoR siempre predice como si fuera mano derecha (espeja la
                    # imagen de la mano izquierda antes de pasarla a la red). Para
                    # recuperar coordenadas reales hay que deshacer ese espejo en
                    # el MISMO orden que usa demo.py oficial: sobre pred_cam antes
                    # de proyectar a la imagen completa, y sobre joints_3d en X,
                    # ambos ANTES de calcular la proyección 2D (no después, como
                    # hacía la versión anterior de este notebook).
                    cam_n = pred_cam[n:n+1].copy()
                    cam_n[:, 1] = multiplier * cam_n[:, 1]

                    # foco real del modelo (config: FOCAL_LENGTH=5000, IMAGE_SIZE=256),
                    # NO un factor fijo de 2 — con foco mal escalado los puntos 2D quedan
                    # desplazados de donde estan las manos en la imagen (bug encontrado
                    # comparando visualmente el overlay contra el video real).
                    scaled_focal = (model_cfg.EXTRA.FOCAL_LENGTH / model_cfg.MODEL.IMAGE_SIZE) * max(img_size_batch[n])
                    cam_t = cam_crop_to_full(
                        cam_n,
                        box_center[n:n+1],
                        box_size[n:n+1],
                        img_size_batch[n:n+1],
                        focal_length=scaled_focal
                    ).squeeze(0)  # (3,)

                    joints_3d[:, 0] = multiplier * joints_3d[:, 0]

                    # FIX (encontrado 2026-09-18, ver notebook Celda 12): cam_crop_to_full
                    # asume que el origen local (0,0,z) cae en el centro del bbox, pero la
                    # muñeca (joint 0) no está en ese origen — wrist_local_x ~ 0.096, no ~0.
                    # Sin restarlo, las dos manos de una imagen se desplazan una hacia la
                    # otra (colapsan al centro) porque el offset tiene signo opuesto en cada
                    # mano (se invierte junto con multiplier). Confirmado visualmente: sin
                    # este resta, las manos quedaban superpuestas entre sí en vez de sobre
                    # su propia posición real.
                    joints_3d = joints_3d - joints_3d[0:1, :]

                    # Proyectar joints 3D (ya espejados y recentrados) a píxeles 2D
                    j3d_cam = joints_3d + cam_t[np.newaxis, :]  # (21, 3) en espacio de cámara
                    cx, cy  = img_size_batch[n, 0] / 2, img_size_batch[n, 1] / 2
                    f       = scaled_focal
                    kpts_2d = np.stack([
                        j3d_cam[:, 0] / j3d_cam[:, 2] * f + cx,
                        j3d_cam[:, 1] / j3d_cam[:, 2] * f + cy,
                    ], axis=1)  # (21, 2) en píxeles

                    row = joints_to_mediapipe_row(joints_3d, kpts_2d, img_w, img_h)

                    if is_right:
                        row_R = row
                    else:
                        row_L = row

        rows_L.append(row_L)
        rows_R.append(row_R)

    cap.release()
    return rows_L, rows_R


print('Función process_video lista.')


## Celda 8 — Ejecutar procesamiento de todos los videos

In [ ]:
errors = []

for video_name in tqdm(videos, desc='Procesando videos'):
    video_id = Path(video_name).stem  # ej: '0000_0000_0000'

    out_L = f'{OUTPUT_DIR}/LEFT_HAND_LANDMARKS/{video_id}.csv'
    out_R = f'{OUTPUT_DIR}/RIGHT_HAND_LANDMARKS/{video_id}.csv'

    # Saltar si ya fue procesado
    if os.path.exists(out_L) and os.path.exists(out_R):
        continue

    try:
        rows_L, rows_R = process_video(f'{VIDEOS_DIR}/{video_name}')

        pd.DataFrame(rows_L, columns=CSV_COLS).to_csv(out_L, index=False)
        pd.DataFrame(rows_R, columns=CSV_COLS).to_csv(out_R, index=False)

    except Exception as e:
        errors.append((video_name, str(e)))
        print(f'ERROR en {video_name}: {e}')

print(f'\nCompletado. Errores: {len(errors)}')
if errors:
    for v, e in errors:
        print(f'  {v}: {e}')

## Celda 9 — Verificación visual

Compara un CSV generado por WiLoR con el original de MediaPipe para una seña.

In [ ]:
import matplotlib.pyplot as plt

TEST_ID = '0007_0000_0000'  # seña a verificar

# WiLoR (generado)
df_wilor = pd.read_csv(f'{OUTPUT_DIR}/RIGHT_HAND_LANDMARKS/{TEST_ID}.csv')

# MediaPipe original — ajusta esta ruta a donde subiste los CSVs de backup
MP_ORIGINAL_DIR = '/content/drive/MyDrive/lsc50/HANDS_MEDIAPIPE'
mp_path = f'{MP_ORIGINAL_DIR}/RIGHT_HAND_LANDMARKS/{TEST_ID}.csv'
df_mp = pd.read_csv(mp_path) if os.path.exists(mp_path) else None
if df_mp is None:
    print(f'CSV de MediaPipe no encontrado en {mp_path}\nSolo se mostrará WiLoR.')

connections = [(0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),
               (0,9),(9,10),(10,11),(11,12),(0,13),(13,14),(14,15),(15,16),
               (0,17),(17,18),(18,19),(19,20)]

def plot_hand(ax, df, frame_idx, title):
    row = df.iloc[frame_idx]
    xs = [row[f'landmark_{i}_x'] for i in range(21)]
    ys = [row[f'landmark_{i}_y'] for i in range(21)]
    ax.scatter(xs, ys, c=range(21), cmap='rainbow', s=60, zorder=3)
    for i in range(21):
        ax.annotate(str(i), (xs[i], ys[i]), fontsize=6)
    for a, b in connections:
        ax.plot([xs[a], xs[b]], [ys[a], ys[b]], 'gray', lw=0.8)
    ax.invert_yaxis()
    ax.set_title(title)
    ax.set_aspect('equal')

mid_frame = len(df_wilor) // 2
n_cols = 3 if df_mp is not None else 2
fig, axes = plt.subplots(1, n_cols, figsize=(7 * n_cols, 6))

# Scatter WiLoR
plot_hand(axes[0], df_wilor, mid_frame, f'WiLoR — frame {mid_frame}')

# Scatter MediaPipe (si está disponible)
if df_mp is not None:
    plot_hand(axes[1], df_mp, mid_frame, f'MediaPipe — frame {mid_frame}')

# Trayectorias X comparadas
ax = axes[-1]
frames = range(len(df_wilor))
ax.plot(frames, df_wilor['landmark_0_x'],  color='steelblue', label='Muñeca WiLoR')
ax.plot(frames, df_wilor['landmark_8_x'],  color='deepskyblue', label='Índice TIP WiLoR')
if df_mp is not None:
    n = min(len(df_wilor), len(df_mp))
    ax.plot(range(n), df_mp['landmark_0_x'].iloc[:n], '--', color='orange', label='Muñeca MP')
    ax.plot(range(n), df_mp['landmark_8_x'].iloc[:n], '--', color='red',    label='Índice TIP MP')
ax.set_title('Trayectoria X por frame')
ax.set_xlabel('Frame')
ax.legend()

plt.suptitle(f'Seña {TEST_ID}', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Frames totales: {len(df_wilor)}')
print(f'Detecciones WiLoR:     {(df_wilor["landmark_0_x"] != 0).sum()}')
if df_mp is not None:
    print(f'Detecciones MediaPipe: {(df_mp["landmark_0_x"] != 0).sum()}')


## Celda 9b — Corregir z en CSVs ya generados

Ejecutar solo si ya procesaste videos antes de corregir la normalización de z.

In [ ]:
# ── Post-proceso: corregir z en CSVs ya generados ────────────────────────────
# Si ya procesaste videos con la normalización incorrecta (z / img_w),
# este script corrige los CSVs existentes multiplicando las columnas z × img_w.
#
# Ejecutar UNA SOLA VEZ. Detecta automáticamente si el CSV ya fue corregido.

import pandas as pd
from pathlib import Path
from tqdm import tqdm

# Resolución de los videos (verificar con cv2.VideoCapture si varía por video)
IMG_W = 640   # ← ajustar si tus videos tienen otra resolución horizontal

Z_COLS = [f'landmark_{i}_z' for i in range(21)]

def fix_csv_z(csv_path):
    df = pd.read_csv(csv_path)
    # Detectar si ya fue corregido: z correctos son ~0.001-0.1, incorrectos ~1e-6
    sample_z = df[Z_COLS].abs().max().max()
    if sample_z > 0.0005:
        return False  # ya corregido o vacío — saltar
    df[Z_COLS] *= IMG_W
    df.to_csv(csv_path, index=False)
    return True

fixed = 0
for side in ['LEFT_HAND_LANDMARKS', 'RIGHT_HAND_LANDMARKS']:
    csvs = list(Path(f'{OUTPUT_DIR}/{side}').glob('*.csv'))
    for csv_path in tqdm(csvs, desc=f'Corrigiendo {side}'):
        if fix_csv_z(csv_path):
            fixed += 1

print(f'CSVs corregidos: {fixed}')
print('(Los que mostraron 0 ya estaban en escala correcta o no tenían detecciones.)')


## Celda 10 — Usar CSVs en el proyecto SignAI

Una vez descargados los CSVs de Google Drive:

```bash
# Hacer backup de los originales
cp -r data/LANDMARKS/HANDS_LANDMARKS data/LANDMARKS/HANDS_LANDMARKS_mediapipe_backup

# Reemplazar con los WiLoR
cp -r /ruta/descarga/HANDS_WILOR/LEFT_HAND_LANDMARKS  data/LANDMARKS/HANDS_LANDMARKS/
cp -r /ruta/descarga/HANDS_WILOR/RIGHT_HAND_LANDMARKS data/LANDMARKS/HANDS_LANDMARKS/
```

El servidor Flask y el pipeline Three.js no requieren ningún cambio.

### Notas de calibración post-proceso

Si los dedos se ven correctos pero la escala de Z hace que la orientación de la mano  
se vea rara, ajustar en `app/static/app.js`:

```js
const FINGER_Z_SCALE = 5.0;  // subir si los dedos no se ven curvados en profundidad
```

### Si WiLoR no detecta manos en algunos frames

Los frames sin detección quedan en cero (igual que MediaPipe). El sistema de freeze  
(`isHandDegenerate`) ya detecta y maneja esto automáticamente.

# Apéndice — Diagnóstico y corrección del desfase de posición

Todo lo que sigue es la investigación que llevó al fix de la Celda 7 (recentrar los joints
respecto a la muñeca antes de proyectar). **No hace falta correr nada de esto para generar
el corpus** — el fix ya está aplicado arriba. Queda como referencia por si el desfase
reaparece con otro tipo de video/cámara, o si se quiere volver a verificar contra el
pipeline oficial de WiLoR.

## Apéndice A — Validar el desfase con `demo.py` OFICIAL (pyrender + GPU)

**Por qué esta celda:** en la Celda 9 se vio que los landmarks reproyectados a CSV (nuestra
Celda 14, sin `pyrender`) quedan desfasados ~50-90px respecto a las manos reales, incluso en
frames fáciles sin oclusión. No se pudo determinar si es un bug en nuestra reproyección manual
o si es así como sale el modelo, porque nunca se corrió el pipeline oficial con el renderizador
real (`pyrender`) que sí monta la malla 3D completa sobre la imagen.

Esta celda corre `demo.py` **tal cual viene del repo, sin ninguna modificación**, sobre el mismo
frame de prueba (ABUELO, video `0018_0000_0000`, frame 0) que se usó para detectar el desfase.

**Cómo leer el resultado:**
- Si la malla 3D calza bien sobre las manos reales → el modelo está bien; el bug está en
  nuestra Celda 7 (función `process_video`, reproyección a CSV) — seguir ahí, revisando cómo se combina `cam_t` con
  `joints_3d` antes de proyectar a píxeles.
- Si la malla también sale desfasada en el render oficial → es comportamiento esperable de
  WiLoR con esta configuración de detección/bbox, y hay que recalibrar cómo interpretamos
  `pred_keypoints_3d` / `pred_cam` (esto fue lo que terminó pasando — ver Apéndice B).

**Requiere:** haber corrido las Celdas 1, 1b, 2 y 3 (repo clonado, dependencias instaladas,
pesos descargados, PKLs de MANO subidos). NO depende de haber corrido las Celdas 4-10 del
pipeline custom.

In [ ]:
# Las dependencias (pyrender, smplx, pytorch-lightning, yacs, chumpy) ya se
# instalaron arriba en la Celda 1b — no hace falta repetirlo acá.

### Subir el frame de prueba

Sube el archivo `0018_0000_0000_frame49.png` (frame 49 de 98 — la mitad del video de
ABUELO, generado localmente con `ffmpeg` desde `app/video_cache/0018_0000_0000.mp4`).

**Nota:** se usa el frame del medio y no el frame 0 porque el frame 0 de una seña suele
ser la pose de reposo (manos abajo/quietas antes de empezar) — el detector YOLO de manos
no encontró nada ahí (`conf=0.3`) y `demo.py` simplemente lo salta sin avisar (`if len(bboxes)
== 0: continue`). A mitad del video las manos ya están levantadas hacienda la seña.

In [ ]:
from google.colab import files
import os, shutil

os.makedirs('demo_test_img', exist_ok=True)
print('Sube 0018_0000_0000_frame49.png:')
uploaded = files.upload()
for fname in uploaded:
    shutil.move(fname, f'demo_test_img/{fname}')
print('Listo:', os.listdir('demo_test_img'))

### (Opcional) Diagnóstico rápido: ¿el detector ve manos en este frame?

Corre esto ANTES de la celda de `demo.py` si quieres confirmar en 10 segundos que el
detector sí encuentra manos en el frame subido, en vez de esperar a que `demo.py` corra
todo el pipeline y te enteres al final por una carpeta vacía. Si `detecciones` da 0,
prueba con otro frame del video (edita `FRAME_TO_CHECK`) antes de seguir.

In [ ]:
from ultralytics import YOLO
import cv2, glob

_diag_detector = YOLO('./pretrained_models/detector.pt')
FRAME_TO_CHECK = glob.glob('demo_test_img/*')[0]

img_bgr = cv2.imread(FRAME_TO_CHECK)
print('Imagen:', FRAME_TO_CHECK, '— shape:', img_bgr.shape)

for conf_try in [0.3, 0.1, 0.05]:
    out = _diag_detector(img_bgr, conf=conf_try, verbose=False)[0]
    print(f'conf={conf_try}: {len(out.boxes)} detecciones', 
          [round(float(c), 3) for c in out.boxes.conf] if len(out.boxes) else '')

### Asegurar que los PKL de MANO también están en `mano_data/` (raíz)

La Celda 5 copió los PKL a `mano_data/mano/` porque así los busca nuestro `model_cfg` cargado
a mano en la Celda 5. El `load_wilor()` **oficial** que usa `demo.py` en cambio espera
`mano_data/MANO_RIGHT.pkl` / `MANO_LEFT.pkl` directamente en la raíz de `mano_data/`
(`model_cfg.MANO.MODEL_PATH = './mano_data/'` dentro de `wilor/models/__init__.py`).
Esta celda copia lo que ya subiste a la ubicación que espera `demo.py`, sin duplicar la subida.

In [ ]:
import os, shutil

os.makedirs('mano_data', exist_ok=True)
for pkl in ['MANO_RIGHT.pkl', 'MANO_LEFT.pkl']:
    dst = f'mano_data/{pkl}'
    if os.path.exists(dst):
        continue
    src = f'mano_data/mano/{pkl}'
    if os.path.exists(src):
        shutil.copy(src, dst)
    else:
        print(f'FALTA {pkl} — vuelve a la Celda 5 (Celda 3 original) y súbelo.')

print('mano_data/:', os.listdir('mano_data'))

### Correr `demo.py` oficial, sin modificar

In [ ]:
!python demo.py --img_folder demo_test_img --out_folder demo_test_out --save_mesh --rescale_factor 2.0

import os
print('Salida:', os.listdir('demo_test_out'))

### Comparar visualmente: frame real vs render oficial de WiLoR

In [ ]:
import matplotlib.pyplot as plt
import cv2, os

import glob
orig_path = glob.glob('demo_test_img/*')[0]  # el que hayas subido, sea cual sea su nombre exacto
orig = cv2.cvtColor(cv2.imread(orig_path), cv2.COLOR_BGR2RGB)

render_files = [f for f in os.listdir('demo_test_out') if f.lower().endswith('.jpg')]
assert render_files, 'demo.py no generó ningún .jpg en demo_test_out — revisar el log de la celda anterior.'
official = cv2.cvtColor(cv2.imread(f'demo_test_out/{render_files[0]}'), cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].imshow(orig)
axes[0].set_title('Frame real — ABUELO 0018, frame 49')
axes[0].axis('off')
axes[1].imshow(official)
axes[1].set_title('Render oficial WiLoR (demo.py + pyrender, sin modificar)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

print('Si la malla calza sobre las manos reales -> el bug está en nuestra Celda 14 (reproyección a CSV).')
print('Si la malla también sale desfasada aquí -> es comportamiento del modelo, recalibrar la Celda 14.')

### Zoom a la región de las manos

El fondo distinto (bosque en vez de fondo azul) es casi seguro un artefacto de pyrender/EGL
en Colab — `wilor/utils/renderer.py` sí pide `bg_color` con alpha 0 (transparente), y el
cuerpo/ropa/cara ya salen idénticos a la foto real, así que el compositing SÍ está usando tu
frame. Eso no afecta lo que nos importa acá: dónde caen los puntos de la mano.

A escala de cuerpo completo un desfase de 50-90px no se nota — hay que recortar la zona de
las manos. Ajusta `CROP_FRAC` (fracción del ancho/alto de la imagen) si las manos no quedan
centradas en el recorte.

In [ ]:
# Fracción aproximada de la imagen donde caen las manos en este video LSC50
# (persona centrada, manos a la altura del torso/cintura). Ajustar si no calza.
CROP_FRAC = dict(x0=0.35, x1=0.65, y0=0.60, y1=0.95)  # bajado — las manos quedaban cortadas en el borde inferior

h, w = orig.shape[:2]
x0, x1 = int(CROP_FRAC['x0'] * w), int(CROP_FRAC['x1'] * w)
y0, y1 = int(CROP_FRAC['y0'] * h), int(CROP_FRAC['y1'] * h)

orig_crop = orig[y0:y1, x0:x1]
official_crop = official[y0:y1, x0:x1]

fig, axes = plt.subplots(1, 2, figsize=(14, 10))
axes[0].imshow(orig_crop)
axes[0].set_title('Frame real — zoom manos')
axes[0].axis('off')
axes[1].imshow(official_crop)
axes[1].set_title('Render oficial WiLoR — zoom manos')
axes[1].axis('off')
plt.tight_layout()
plt.show()

### ¿El bbox de YOLO ya está mal ubicado, o es la reproyección de WiLoR?

El desfase visto arriba aparece con el pipeline OFICIAL sin modificar — o sea, no es un bug
de nuestra Celda 14. Falta separar dos posibles causas dentro del propio WiLoR:

1. El detector YOLO ya entrega un cuadro (bbox) descentrado de la mano real (problema de
   detección en imágenes de cuerpo completo, donde la mano es chica).
2. El bbox está bien, pero la malla 3D se proyecta mal DENTRO de ese cuadro (problema de
   `cam_crop_to_full` / escala de foco con esta configuración).

Esta celda dibuja el bbox crudo de YOLO sobre el frame real. Si el cuadro ya queda corrido
respecto a la mano real, la causa es (1); si el cuadro cae bien sobre la mano pero el guante
renderizado queda desplazado DENTRO de esa región, la causa es (2).

In [ ]:
import cv2, glob
import matplotlib.pyplot as plt
from ultralytics import YOLO

_bbox_detector = YOLO('./pretrained_models/detector.pt')
test_img_path = glob.glob('demo_test_img/*')[0]
img_bgr = cv2.imread(test_img_path)

det = _bbox_detector(img_bgr, conf=0.3, verbose=False)[0]
img_draw = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).copy()

for box in det.boxes:
    x0, y0, x1, y1 = box.xyxy[0].cpu().numpy().astype(int)
    conf = float(box.conf[0])
    is_right = int(box.cls[0])
    label = f"{'right' if is_right == 1 else 'left'} {conf:.2f}"
    color = (255, 0, 0) if is_right == 1 else (0, 200, 0)
    cv2.rectangle(img_draw, (x0, y0), (x1, y1), color, 3)
    cv2.putText(img_draw, label, (x0, max(0, y0 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

# Zoom a la misma región que la celda de comparación, para ver el bbox junto a la mano real
h, w = img_draw.shape[:2]
x0c, x1c = int(CROP_FRAC['x0'] * w), int(CROP_FRAC['x1'] * w)
y0c, y1c = int(CROP_FRAC['y0'] * h), int(CROP_FRAC['y1'] * h)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(img_draw)
axes[0].set_title('bbox de YOLO sobre el frame completo')
axes[0].axis('off')
axes[1].imshow(img_draw[y0c:y1c, x0c:x1c])
axes[1].set_title('Zoom al bbox (misma región que el zoom de manos)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## Apéndice B — Comparar NUESTRA reproyección custom contra el frame real

Con el recorte bien puesto, el render OFICIAL de `demo.py` sí calza sobre las manos reales
— o sea, el desfase de ayer no viene de WiLoR ni de la fórmula de `cam_crop_to_full`, sino
de algo específico de nuestra propia Celda 7 (`process_video`, la que genera los CSVs del
corpus).

Diferencia encontrada al comparar código: nuestra Celda 7 convierte el frame a RGB antes de
pasarlo al detector YOLO y al `ViTDetDataset`
(`img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)`); `demo.py` oficial pasa la imagen BGR
cruda (de `cv2.imread`) sin convertir a ninguno de los dos. Si el modelo espera BGR, esa
conversión de más sería justo la causa del desfase.

**Requiere haber corrido antes en esta sesión:** Celda 5 (carga el modelo `model`/`model_cfg`
a mano) y Celda 6 (funciones `cam_crop_to_full`/`joints_to_mediapipe_row` custom). No hace
falta Celda 4 (Drive) ni Celdas 7-10 (esas son para procesar el corpus completo).

In [ ]:
import glob, cv2
import numpy as np
import matplotlib.pyplot as plt

BATCH_SIZE = 8       # igual que en la Celda 7 (no depende de haberla corrido)
RESCALE_FACTOR = 2.0  # idem

test_img_path = glob.glob('demo_test_img/*')[0]
frame_bgr = cv2.imread(test_img_path)
img_h, img_w = frame_bgr.shape[:2]


def run_custom_reprojection(img_for_model, recenter_wrist=False):
    """Corre detector + WiLoR + nuestra reproyección custom (Celda 12/14) sobre
    la imagen que se le pase, y devuelve una lista de (kpts_2d, is_right) por mano
    detectada. img_for_model determina el orden de canales que ve el modelo.

    recenter_wrist=True aplica el fix encontrado: cam_crop_to_full asume que el
    origen local (0,0,z) cae en el centro del bbox, pero la muñeca (joint 0) no
    está en el origen local (wrist_local_x ~ 0.096, no ~0) — se resta joints_3d[0]
    de todos los joints ANTES de sumar cam_t para corregirlo."""
    det_out = detector(img_for_model, verbose=False)
    boxes = det_out[0].boxes
    results = []
    if boxes is None or len(boxes) == 0:
        return results

    bboxes = boxes.xyxy.cpu().numpy()
    is_right_arr = boxes.cls.cpu().numpy().astype(int)
    print('  bboxes (x0,y0,x1,y1):')
    for bb, ir in zip(bboxes, is_right_arr):
        print(f'    is_right={ir}  box={bb.round(1).tolist()}  centro_bbox=({(bb[0]+bb[2])/2:.1f}, {(bb[1]+bb[3])/2:.1f})')

    dataset = ViTDetDataset(model_cfg, img_for_model, bboxes, right=is_right_arr, rescale_factor=RESCALE_FACTOR)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    for batch in dataloader:
        batch = recursive_to(batch, device)
        with torch.no_grad():
            out = model(batch)

        pred_keypoints_3d = out['pred_keypoints_3d'].cpu().numpy()
        pred_cam          = out['pred_cam'].cpu().numpy()
        is_right_batch    = batch['right'].cpu().numpy()
        box_center        = batch['box_center'].cpu().numpy()
        box_size          = batch['box_size'].cpu().numpy()
        img_size_batch    = batch['img_size'].cpu().numpy()

        for n in range(len(pred_keypoints_3d)):
            joints_3d  = pred_keypoints_3d[n].copy()
            is_right   = bool(is_right_batch[n])
            multiplier = 1.0 if is_right else -1.0

            cam_n = pred_cam[n:n+1].copy()
            cam_n[:, 1] = multiplier * cam_n[:, 1]

            scaled_focal = (model_cfg.EXTRA.FOCAL_LENGTH / model_cfg.MODEL.IMAGE_SIZE) * max(img_size_batch[n])
            cam_t = cam_crop_to_full(
                cam_n, box_center[n:n+1], box_size[n:n+1], img_size_batch[n:n+1],
                focal_length=scaled_focal
            ).squeeze(0)

            joints_3d[:, 0] = multiplier * joints_3d[:, 0]
            if recenter_wrist:
                joints_3d = joints_3d - joints_3d[0:1, :]
            j3d_cam = joints_3d + cam_t[np.newaxis, :]
            cx, cy  = img_size_batch[n, 0] / 2, img_size_batch[n, 1] / 2
            f       = scaled_focal
            kpts_2d = np.stack([
                j3d_cam[:, 0] / j3d_cam[:, 2] * f + cx,
                j3d_cam[:, 1] / j3d_cam[:, 2] * f + cy,
            ], axis=1)
            print(f'  n={n} is_right={is_right}  box_center={box_center[n].round(1).tolist()}  '
                  f'box_size={float(box_size[n]):.1f}  cam_t={cam_t.round(4).tolist()}  '
                  f'kpts_2d_centro={kpts_2d.mean(axis=0).round(1).tolist()}  '
                  f'wrist_local_xyz={joints_3d[0].round(4).tolist()}')
            results.append((kpts_2d, is_right))
    return results


# Variante B: como demo.py oficial (BGR crudo, sin convertir), sin el fix
print('--- Variante B: BGR crudo, SIN recentrar muñeca (la que colapsaba) ---')
results_bgr = run_custom_reprojection(frame_bgr, recenter_wrist=False)

# Variante C: BGR crudo + fix de recentrado en la muñeca
print('--- Variante C: BGR crudo, CON recentrado en la muñeca (fix) ---')
results_fixed = run_custom_reprojection(frame_bgr, recenter_wrist=True)

print(f'Variante B (sin fix): {len(results_bgr)} manos')
print(f'Variante C (con fix): {len(results_fixed)} manos')

connections = [(0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),
               (0,9),(9,10),(10,11),(11,12),(0,13),(13,14),(14,15),(15,16),
               (0,17),(17,18),(18,19),(19,20)]

def plot_overlay(ax, base_rgb_img, results, title, x0, y0):
    ax.imshow(base_rgb_img)
    for kpts_2d, is_right in results:
        xs, ys = kpts_2d[:, 0] - x0, kpts_2d[:, 1] - y0
        color = 'red' if is_right else 'lime'
        for a, b in connections:
            ax.plot([xs[a], xs[b]], [ys[a], ys[b]], color=color, lw=1.5)
        ax.scatter(xs, ys, c=color, s=15, zorder=3)
    ax.set_title(title)
    ax.axis('off')

x0c, x1c = int(CROP_FRAC['x0'] * img_w), int(CROP_FRAC['x1'] * img_w)
y0c, y1c = int(CROP_FRAC['y0'] * img_h), int(CROP_FRAC['y1'] * img_h)
base_crop = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)[y0c:y1c, x0c:x1c]

fig, axes = plt.subplots(1, 2, figsize=(14, 8))
plot_overlay(axes[0], base_crop, results_bgr, 'SIN fix (colapsa al centro)', x0c, y0c)
plot_overlay(axes[1], base_crop, results_fixed, 'CON fix (recentrado en la muñeca)', x0c, y0c)
plt.tight_layout()
plt.show()

## Apéndice C — Validar el fix en 2-3 señas más (ya hecho, referencia)

Antes de lanzar el fix sobre las 1000 señas (Celda 8), confirmar que generaliza más allá de
ABUELO. Se preparan dos frames más de la prueba piloto original (TÍO y HOLA):

- **TÍO** (`0019_0000_0000`, frame 41): manos juntas/entrelazadas — buen caso de contacto,
  ya que esta seña tiene ~49% de frames en CONTACT según el análisis de `hand_corrector.py`.
- **HOLA** (`0047_0000_0000`, frame 37): manos separadas y levantadas — pose muy distinta a
  ABUELO (altura, orientación), buen caso "fácil" para confirmar que el fix no depende de la
  pose específica de ABUELO.

**Requiere haber corrido antes en esta sesión:** Celdas 1-3, Celda 5, Celda 6, y la Celda 12
completa (de ahí se reutiliza `run_custom_reprojection` ya con el fix).

In [ ]:
from google.colab import files
import os, shutil

os.makedirs('validation_imgs', exist_ok=True)
print('Sube 0019_0000_0000_frame41_para_colab.png y 0047_0000_0000_frame37_para_colab.png (los dos juntos):')
uploaded = files.upload()
for fname in uploaded:
    shutil.move(fname, f'validation_imgs/{fname}')
print('Listo:', os.listdir('validation_imgs'))

In [ ]:
import glob, cv2
import numpy as np
import matplotlib.pyplot as plt

val_paths = sorted(glob.glob('validation_imgs/*'))
assert val_paths, 'No hay imágenes en validation_imgs/ — corre la celda de subida primero.'

connections = [(0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),
               (0,9),(9,10),(10,11),(11,12),(0,13),(13,14),(14,15),(15,16),
               (0,17),(17,18),(18,19),(19,20)]

fig, axes = plt.subplots(1, len(val_paths), figsize=(9 * len(val_paths), 8))
if len(val_paths) == 1:
    axes = [axes]

for ax, path in zip(axes, val_paths):
    frame_bgr = cv2.imread(path)
    print(f'--- {os.path.basename(path)} ---')
    results = run_custom_reprojection(frame_bgr, recenter_wrist=True)

    # Recorte automático: unión de todos los bboxes/keypoints detectados + margen,
    # en vez de un CROP_FRAC fijo (cada seña tiene las manos en una zona distinta).
    all_xy = np.concatenate([kpts for kpts, _ in results], axis=0)
    x0, y0 = all_xy.min(axis=0) - 80
    x1, y1 = all_xy.max(axis=0) + 80
    x0, y0 = max(0, int(x0)), max(0, int(y0))
    x1, y1 = min(frame_bgr.shape[1], int(x1)), min(frame_bgr.shape[0], int(y1))

    base_crop = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)[y0:y1, x0:x1]
    ax.imshow(base_crop)
    for kpts_2d, is_right in results:
        xs, ys = kpts_2d[:, 0] - x0, kpts_2d[:, 1] - y0
        color = 'red' if is_right else 'lime'
        for a, b in connections:
            ax.plot([xs[a], xs[b]], [ys[a], ys[b]], color=color, lw=1.5)
        ax.scatter(xs, ys, c=color, s=15, zorder=3)
    ax.set_title(os.path.basename(path))
    ax.axis('off')

plt.tight_layout()
plt.show()